In [1]:
# Domain layer: Pure business logic for articles and summarization
from dataclasses import dataclass
from typing import Optional
import nltk
from nltk.tokenize import sent_tokenize

# Download NLTK resources (run once)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

@dataclass
class Article:
    """Represents a news article with cleaned and summarized content."""
    title: str
    summary: str
    link: str
    pub_date: str
    thumbnail: Optional[str]

def simple_summarize(text: str, max_sentences: int = 2) -> str:
    """Summarize text by extracting the first few sentences."""
    sentences = sent_tokenize(text)
    summary = ' '.join(sentences[:min(max_sentences, len(sentences))])
    return summary.strip()

In [5]:
# Infrastructure layer: Fetch and clean RSS feeds
import feedparser
from bs4 import BeautifulSoup
from typing import List, Optional
from /domain.article import Article

def clean_text(html_text: str) -> str:
    """Remove HTML tags and clean text."""
    if not html_text:
        return ""
    soup = BeautifulSoup(html_text, 'html.parser')
    text = soup.get_text(separator=' ', strip=True)
    return text

def fetch_rss_feed(feed_url: str, max_articles: int = 5) -> List[Article]:
    """Fetch and parse an RSS feed into a list of Articles."""
    try:
        feed = feedparser.parse(feed_url)
        articles = []
        for entry in feed.entries[:max_articles]:
            title = entry.get('title', '')
            description = clean_text(entry.get('description', '') or entry.get('summary', ''))
            link = entry.get('link', '')
            pub_date = entry.get('published', '') or entry.get('updated', '')
            # Extract thumbnail (varies by feed)
            thumbnail = None
            if 'media_thumbnail' in entry:
                thumbnail = entry.media_thumbnail[0].get('url')
            elif 'media_content' in entry:
                for content in entry.media_content:
                    if content.get('medium') == 'image':
                        thumbnail = content.get('url')
                        break
            articles.append(Article(
                title=title,
                summary=description,  # Will be summarized later
                link=link,
                pub_date=pub_date,
                thumbnail=thumbnail
            ))
        return articles
    except Exception as e:
        print(f"Error fetching feed {feed_url}: {e}")
        return []

SyntaxError: invalid syntax (2470591742.py, line 5)

In [2]:
# Application layer:  RSS feed processing
from typing import List
from domain.article import Article, simple_summarize
from infrastructure.rss_repository import fetch_rss_feed

def process_rss_feed(feed_url: str, max_articles: int = 5) -> List[Article]:
    """Fetch, clean, and summarize articles from an RSS feed."""
    articles = fetch_rss_feed(feed_url, max_articles)
    # Summarize descriptions
    for article in articles:
        article.summary = simple_summarize(article.summary, max_sentences=2)
    return articles

ModuleNotFoundError: No module named 'domain'

## Testing step 3

In [7]:
# Set up project path
import sys
import os
sys.path.append(os.path.abspath('..'))

In [4]:
# Test RSS feed parsing
from application.feed_service import process_rss_feed

# Define feed URLs
feeds = {
    'skynews': 'https://feeds.skynews.com/feeds/rss/home.xml',
    'bbc': 'https://feeds.bbci.co.uk/news/rss.xml',
    'guardian': 'https://www.theguardian.com/world/rss'
}

# Test Sky News
articles = process_rss_feed(feeds['skynews'], max_articles=2)
for article in articles:
    print(f"Title: {article.title}")
    print(f"Summary: {article.summary}")
    print(f"Link: {article.link}")
    print(f"Pub Date: {article.pub_date}")
    print(f"Thumbnail: {article.thumbnail}")
    print("-" * 80)

Title: Why have tensions escalated between India and Pakistan?
Summary: India has launched a missile attack on Pakistan and the territory Islamabad administers in Kashmir, killing at least 26 civilians, Pakistani officials have said.
Link: https://news.sky.com/story/kashmir-terrorist-attack-what-happened-and-how-have-india-and-pakistan-reacted-13355235
Pub Date: Tue, 06 May 2025 16:23:00 +0100
Thumbnail: https://e3.365dm.com/25/05/1920x1080/skynews-kashmir-india-pakistan_6908739.jpg?20250507093912
--------------------------------------------------------------------------------
Title: Cardinals begin voting for a new pope: Here's a look at 11 contenders
Summary: The papal conclave is beginning, where 133 cardinal electors are tasked with choosing the new leader of the Catholic Church.
Link: https://news.sky.com/story/who-could-be-the-next-pope-13311775
Pub Date: Mon, 21 Apr 2025 12:29:00 +0100
Thumbnail: https://e3.365dm.com/25/05/1920x1080/skynews-pope-conclave_6908164.png?202505061528

In [5]:
# Test all feeds
for feed_name, feed_url in feeds.items():
    print(f"\nTesting {feed_name.upper()}:")
    articles = process_rss_feed(feed_url, max_articles=2)
    print(f"Found {len(articles)} articles")
    for article in articles:
        print(f"Title: {article.title}")
        print(f"Summary: {article.summary}")
        print("-" * 80)


Testing SKYNEWS:
Found 2 articles
Title: Why have tensions escalated between India and Pakistan?
Summary: India has launched a missile attack on Pakistan and the territory Islamabad administers in Kashmir, killing at least 26 civilians, Pakistani officials have said.
--------------------------------------------------------------------------------
Title: Cardinals begin voting for a new pope: Here's a look at 11 contenders
Summary: The papal conclave is beginning, where 133 cardinal electors are tasked with choosing the new leader of the Catholic Church.
--------------------------------------------------------------------------------

Testing BBC:
Found 2 articles
Title: What happens next between India and Pakistan? Four key questions
Summary: Experts from India and Pakistan weigh in on what’s next -  and how this clash differs from past conflicts.
--------------------------------------------------------------------------------
Title: Joe Biden on Trump: 'What president ever talks like

In [9]:
# Test FastAPI API
import requests

# Test Sky News API
try:
    response = requests.get("http://127.0.0.1:8000/articles?feed=skynews&max_articles=2")
    response.raise_for_status()
    print("Sky News API Response:")
    for article in response.json():
        print(f"Title: {article['title']}")
        print(f"Summary: {article['summary']}")
        print(f"Link: {article['link']}")
        print(f"Pub Date: {article['pub_date']}")
        print(f"Thumbnail: {article['thumbnail']}")
        print("-" * 80)
except requests.RequestException as e:
    print(f"API Error: {e}")

# Test all feeds
feeds = ['skynews', 'bbc', 'guardian']
for feed in feeds:
    try:
        response = requests.get(f"http://127.0.0.1:8000/articles?feed={feed}&max_articles=2")
        response.raise_for_status()
        print(f"\n{feed.upper()} API Response:")
        print(f"Found {len(response.json())} articles")
        for article in response.json():
            print(f"Title: {article['title']}")
            print(f"Summary: {article['summary']}")
            print("-" * 80)
    except requests.RequestException as e:
        print(f"{feed.upper()} API Error: {e}")

Sky News API Response:
Title: Why many assume interest rates will fall further - but no one really has a clue
Summary: Let's deal, first of all, with the question many of you will have: after today's reduction to 4.25% will there be more interest rate cuts to come?
Link: https://news.sky.com/story/why-many-assume-interest-rates-will-fall-further-but-no-one-really-has-a-clue-13363780
Pub Date: Thu, 08 May 2025 11:39:00 +0100
Thumbnail: https://e3.365dm.com/24/07/1920x1080/skynews-biz-graphic-bank-bailey_6643991.png?20240731155213
--------------------------------------------------------------------------------
Title: King and Queen among thousands at VE Day anniversary
Summary: The King and Queen have paid their respects to Britain's war dead at a service to mark the 80th anniversary of VE Day.
Link: https://news.sky.com/story/king-and-queen-attend-ve-day-80th-anniversary-service-at-westminster-abbey-13363803
Pub Date: Thu, 08 May 2025 12:14:00 +0100
Thumbnail: https://e3.365dm.com/25/05